In [30]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
from pytorch_forecasting import TimeSeriesDataSet, DeepAR
from pytorch_forecasting.metrics import NormalDistributionLoss
from lightning.pytorch import Trainer

In [17]:
df = pd.read_csv(r"C:\Users\cecil\Cript_Anomalies\BTCUSDT_20180101_20260112.csv", delimiter=";", skiprows=0)
display(df)

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,n_trades,taker_buy_base,...,ignore,symbol,interval,log_return,volatility_20,range_hl,trades_per_volume,buy_ratio,z_return,anomaly_simple
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,0,BTCUSDT,15m,NaN,NaN,0.023013,12.716799,0.511480,NaN,False
1,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,0,BTCUSDT,15m,-0.002587,NaN,0.011000,14.887438,0.485919,-0.691986,False
2,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,0,BTCUSDT,15m,-0.003757,NaN,0.007064,12.515012,0.547036,-1.004095,False
3,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,0,BTCUSDT,15m,0.004341,NaN,0.017849,8.433327,0.521511,1.156086,False
4,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,0,BTCUSDT,15m,-0.006182,NaN,0.012526,12.379798,0.472275,-1.650855,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281126,2026-01-12 22:30:00+00:00,91269.52,91402.81,91203.69,91339.90,86.681550,2026-01-12 22:44:59.999000+00:00,7.912440e+06,28813,50.482400,...,0,BTCUSDT,15m,0.000771,0.001837,0.002182,332.400609,0.582389,0.203804,False
281127,2026-01-12 22:45:00+00:00,91339.89,91339.90,91214.00,91244.99,60.451790,2026-01-12 22:59:59.999000+00:00,5.517664e+06,19198,18.105290,...,0,BTCUSDT,15m,-0.001040,0.001788,0.001378,317.575377,0.299500,-0.279123,False
281128,2026-01-12 23:00:00+00:00,91244.99,91280.99,91059.88,91160.07,180.361250,2026-01-12 23:14:59.999000+00:00,1.643558e+07,26239,91.014310,...,0,BTCUSDT,15m,-0.000931,0.001721,0.002423,145.480251,0.504622,-0.250178,False
281129,2026-01-12 23:15:00+00:00,91160.07,91282.39,91135.58,91269.34,31.328540,2026-01-12 23:29:59.999000+00:00,2.857620e+06,9456,19.653920,...,0,BTCUSDT,15m,0.001198,0.001643,0.001610,301.833408,0.627349,0.317736,False


In [ ]:
def prepare_deepar_df(df, time_col="open_time", group_col="symbol"):
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col])
    d = d.sort_values([group_col, time_col]).reset_index(drop=True)
    d["time_idx"] = d.groupby(group_col).cumcount()
    d["hour"] = d[time_col].dt.hour.astype(float)
    d["dayofweek"] = d[time_col].dt.dayofweek.astype(float)
    return d

def fit_deepar(
    df,
    target="log_return",
    time_idx="time_idx",
    time_col="open_time",
    group_ids=["symbol"],
    encoder_len=96,
    pred_len=12,
    batch_size=256,
    max_epochs=8
):
    d = prepare_deepar_df(df, time_col=time_col, group_col=group_ids[0])

    # covariables que queremos que estén en encoder Y decoder (para evitar el assert)
    cov_reals = ["volatility_20", "range_hl", "trades_per_volume", "buy_ratio", "hour", "dayofweek"]

    # limpiar NaN
    d = d.dropna(subset=[target] + cov_reals + [time_idx] + group_ids).copy()

    max_idx = d[time_idx].max()
    cutoff = int(max_idx * 0.8)
    train_df = d[d[time_idx] <= cutoff]
    val_df   = d[d[time_idx] > cutoff]

    training = TimeSeriesDataSet(
        train_df,
        time_idx=time_idx,
        target=target,
        group_ids=group_ids,
        max_encoder_length=encoder_len,
        max_prediction_length=pred_len,

        # ✅ todo lo que queremos en decoder también:
        time_varying_known_reals=cov_reals,

        # ✅ unknown: solo el target (DeepAR lo necesita)
        time_varying_unknown_reals=[target],

        add_relative_time_idx=False,
        add_encoder_length=False,
        add_target_scales=True,   # este generalmente no rompe aquí
        allow_missing_timesteps=True,
    )

    validation = TimeSeriesDataSet.from_dataset(training, val_df, stop_randomization=True)

    train_loader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_loader   = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

    model = DeepAR.from_dataset(
        training,
        learning_rate=1e-3,
        hidden_size=40,
        rnn_layers=2,
        dropout=0.1,
        loss=NormalDistributionLoss(),
    )

    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices="auto",
        enable_progress_bar=True,
        log_every_n_steps=200,
        logger=False,
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    return model, training, validation, d

In [19]:
def score_deepar_anomalies(model, dataset, contamination=0.01):

    loader = dataset.to_dataloader(train=False, batch_size=512)

    preds = model.predict(loader, mode="quantiles", quantiles=[0.1,0.5,0.9])

    q10 = preds[...,0][:,0]
    q50 = preds[...,1][:,0]
    q90 = preds[...,2][:,0]

    df_pred = dataset.to_pandas()

    df_pred["deepar_q10"] = q10
    df_pred["deepar_q50"] = q50
    df_pred["deepar_q90"] = q90

    df_pred["anomaly_deepar"] = (
        (df_pred["log_return"] < df_pred["deepar_q10"]) |
        (df_pred["log_return"] > df_pred["deepar_q90"])
    ).astype(int)

    return df_pred

In [20]:
def plot_close_anomalies(df, close_col="close", anomaly_col="anomaly_dagmm", title="DAGMM - Anomalías"):
    d = df.copy()
    
    plt.figure(figsize=(14,6))
    plt.plot(d[close_col].values, alpha=0.7, label=close_col, color="steelblue")

    an = d[d[anomaly_col] == 1]
    plt.scatter(an.index, an[close_col], 
                s=25, 
                label="anomalía",
                color="purple")

    plt.title(title)
    plt.legend()
    plt.show()

In [21]:
features = [
    "log_return",
    "volatility_20",
    "range_hl",
    "trades_per_volume",
    "buy_ratio"
]

In [22]:
df_clean = df.copy()

# 1) quitar inf
df_clean[features] = df_clean[features].replace([np.inf, -np.inf], np.nan)

# 2) eliminar filas con NA en features
df_clean = df_clean.dropna(subset=features).reset_index(drop=True)

In [ ]:
deepar_model, train_ds, val_ds, df_deepar = fit_deepar(
    df=df_clean,
    encoder_len=96,
    pred_len=12,
    max_epochs=8
)

c:\Users\cecil\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\cecil\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi

In [ ]:
df_scored_deepar = score_deepar_anomalies(
    deepar_model,
    val_ds
)

In [ ]:
plot_close_anomalies(
    df_scored_deepar,
    close_col="close",
    anomaly_col="anomaly_deepar",
    title="Anomalías detectadas con DeepAR"
)